In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import datetime
from matplotlib.backends.backend_pdf import PdfPages
from collections import defaultdict

pd.options.display.max_rows=2000
pd.options.display.max_columns=70


In [ ]:
# Import and prep data
# Data importing stage

# Start with latest (?) cleaned data
cleaned_ema_path = "/Volumes/private/studydata/risk/data_processed/ema/ema.csv"
hourly_labels_path = "/Volumes/private/studydata/risk/data_processed/ema/labels_1hour.csv"
daily_labels_path = "/Volumes/private/studydata/risk/data_processed/ema/labels_1day.csv"
weekly_labels_path = "/Volumes/private/studydata/risk/data_processed/ema/labels_1week.csv"
study_dates_path = "/Volumes/private/studydata/risk/data_processed/ema/study_dates.csv" 

# Data imports
# Ecological momentary assessments
ema_raw = pd.read_csv(cleaned_ema_path,parse_dates=['dttm_obs'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))
# Data labels
hourly_labels_raw = pd.read_csv(hourly_labels_path,parse_dates=['dttm_label'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))
daily_labels_raw = pd.read_csv(daily_labels_path,parse_dates=['dttm_label'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))
weekly_labels_raw = pd.read_csv(weekly_labels_path,parse_dates=['dttm_label'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))
# Background information about the study dates for each individual
study_dates_raw = pd.read_csv(study_dates_path,parse_dates=['study_start','data_start','study_end','ema_end'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))

# Path of the raw list of lapses
lapse_list_path = '/Volumes/private/studydata/risk/data_processed/shared/lapses.csv'
special_lapse_list_path = "/Volumes/private/studydata/risk/data_processed/shared/pulick_excluded_lapse_special_handling_modified.csv"
# Import and copy the dataframe
lapse_import_raw = pd.read_csv(lapse_list_path,parse_dates=['lapse_start','lapse_end'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))
special_lapse_import_raw = pd.read_csv(special_lapse_list_path,parse_dates=['lapse_start','lapse_end'],date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%dT%H:%M:%S",utc=True).tz_convert('US/Central'))
full_import_raw = pd.concat([lapse_import_raw,special_lapse_import_raw],ignore_index=True)
lapse_summary_df = full_import_raw.drop(['ema_end'],axis=1).join(study_dates_raw.set_index('subid'),on='subid')
# Drop miscellaneous columns and get rid of 'exclude'==True rows
lapse_summary_df.drop(['lapse_start_date','lapse_start_time','lapse_end_time','lapse_end_date','response_id','lapse_cnt'],axis=1,inplace=True)
lapse_summary_df.drop(lapse_summary_df.loc[lapse_summary_df['exclude']==True].index, inplace=True)

# Put the duration into days
lapse_summary_df['day_lapse_duration'] = lapse_summary_df.duration/24

# Create relevant columns
lapse_summary_df['td_date']=lapse_summary_df.lapse_start.sub(lapse_summary_df.data_start)
lapse_summary_df['norm_date']=lapse_summary_df['td_date'].dt.days+lapse_summary_df['td_date'].dt.seconds/86400
lapse_summary_df['hour_of_day']=lapse_summary_df.apply(lambda row: row.lapse_start.hour,axis=1)
lapse_summary_df['day_of_week']=lapse_summary_df.apply(lambda row: row.lapse_start.day_name(),axis=1)
lapse_summary_df['day_of_week_num']=lapse_summary_df.apply(lambda row: row.lapse_start.weekday(),axis=1)
lapse_summary_df['time_of_week']=lapse_summary_df.apply(lambda row: row.lapse_start.hour/24+row.day_of_week_num,axis=1)
lapse_summary_df['end_td_date']=lapse_summary_df.lapse_end.sub(lapse_summary_df.data_start)
lapse_summary_df["start_rounded_date"]=lapse_summary_df.apply(lambda row: np.nan if np.isnan(row.norm_date) else int(np.floor(row.norm_date)),axis=1)

# Final cleanup/drop
lapse_summary_df.rename(columns={'duration':'hour_lapse_duration'},inplace=True)
lapse_summary_df.drop(['exclude','ema_1_6','study_start','data_start','study_end','ema_end'],inplace=True,axis=1)

# Use the same list of subids used for the previously generated labels
subid_list = hourly_labels_raw.subid.unique()

# Take a peek
lapse_summary_df.head()

In [ ]:
# Start taking a look at EMA responses
# Shift corresponds to a day that starts and ends at 4AM (with some epsilon factor for numerical issues)
day_shift_typ = 4/24+0.00001
day_shift_alt = 3.5/24+0.00001

def get_rounded_date(val, subject, day_shift_typ, day_shift_alt):
    # Subjects 42 and 43 were early risers, use 3:30AM-3:30AM day for them
    if subject in [42,43]:
        shift = day_shift_alt
    else:
        shift = day_shift_typ
    # Special case for day 1
    if val<(shift+1):
        return 0
    else:
        return np.floor(val-shift)
    
# Join the relevant study date labels
ema_df = ema_raw.join(study_dates_raw.set_index('subid'),on='subid').dropna(subset='ema_2',axis=0)
print(ema_raw.shape)
print(ema_df.shape)
# Drop the folks with no study date information
# 3/22 note -- looks like this doesn't actually drop anyone, commenting out
# ema_df.dropna(axis=0,how='any',subset='study_start',inplace=True)

# Sort the information first by subid then datetime
ema_df.sort_values(['subid','dttm_obs'],inplace=True,ignore_index=True)

# Calculate the time-delta between the start of data and datetime label (to normalize for people studied at different times)
# Testing out swap to study_start
ema_df['td_date']=ema_df.dttm_obs.sub(ema_df.study_start)
#ema_df['td_date']=ema_df.dttm_obs.sub(ema_df.data_start)

# Convert the time-delta to an numeric value (in days)
ema_df['norm_date']=ema_df['td_date'].dt.days+ema_df['td_date'].dt.seconds/86400

# Take the floor of the date to get integer date numbers
ema_df["rounded_date"]=ema_df.apply(lambda row: get_rounded_date(row.norm_date,row.subid,day_shift_typ,day_shift_alt),axis=1)
# Get a numeric time of day
ema_df["time_of_day"]=ema_df.apply(lambda row: (row.norm_date % 1)*24,axis=1)
# Drop study info
ema_df.drop(columns=['study_start','data_start','study_end','ema_end'],axis=0,inplace=True)
# DF for morning EMAs only
mema_df = ema_df.query("ema_type=='morning'").copy(deep=True)
mema_df["ema_2_scaled"]=mema_df["ema_2"]/12
mema_df["ema_3_scaled"]=mema_df["ema_3"]/12
mema_df["ema_4_scaled"]=mema_df["ema_4"]/12
mema_df["ema_5_scaled"]=mema_df["ema_5"]/12
mema_df["ema_6_scaled"]=(mema_df["ema_6"]-1)/10
mema_df["ema_7_scaled"]=(mema_df["ema_7"]-1)/10
mema_df["ema_8_scaled"]=(mema_df["ema_8"]-1)/10
mema_df["ema_9_scaled"]=(mema_df["ema_9"]-1)/10
mema_df["ema_10_scaled"]=(mema_df["ema_10"]-1)/10
#mema_df.drop(columns=['ema_1','ema_1_1','ema_1_2','ema_1_3','ema_1_4','ema_1_5','ema_1_6','ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10'],inplace=True)
mema_df.head()

In [ ]:
ax=sns.histplot(mema_df,x='time_of_day')
ax.set_xticks(list(range(0,26,2)))

In [ ]:
data_range_list = []
for id in ema_df.subid.unique():
    temp_df = ema_df.query("subid==@id & ema_type=='morning'").copy()
    ema_dates = temp_df.norm_date.to_numpy()
    unique_days = len(temp_df.rounded_date.unique())
    earliest = ema_dates.min()
    latest = ema_dates.max()
    iter_df = pd.DataFrame({'subid':id,'response_days':unique_days,
                            'earliest':earliest,'latest':latest},index=[0])
    data_range_list.append(iter_df)

data_range_df = pd.concat(data_range_list,ignore_index=True)
data_range_df.head()

In [ ]:
data_range_df.query("latest >= 90")

In [ ]:
ax=sns.histplot(data_range_df,x='latest')

In [ ]:
# Create a day-level dataset (day is labeled 1 if a lapse starts on that day, 0 otherwise)
label_list=[]
subid_info_list=[]
duplicate_ema_counter = 0
# Get the subids from the labels dataset that was originally imported
subids=hourly_labels_raw.subid.unique()
# Loop over the subjects
for id in subids:
    # Special handling of early risers (3:30 day cutoff rather than 4)
    if id in [42,43]:
        day_shift = day_shift_alt
    else:
        day_shift = day_shift_typ
    # Identify the day of the week for day 0 for this particular patient
    data_start_day_of_week=study_dates_raw.query("subid==@id").data_start.iloc[0].weekday()

    # Get start/end dates for the subject
    subid_study_dates = study_dates_raw.query("subid==@id")
    id_study_start = subid_study_dates.study_start
    id_data_start = subid_study_dates.data_start
    id_study_end = subid_study_dates.study_end
    id_data_end = subid_study_dates.ema_end

    # EMA
    subid_mema = mema_df.query("subid==@id").copy()
    mema_days = subid_mema.rounded_date.unique()

    first_morning_ema = subid_mema.dttm_obs.min()
    first_morning_ema_rescaled = subid_mema.loc[subid_mema.dttm_obs==first_morning_ema].norm_date.values[0]
    #print(first_morning_ema)
    #print(first_morning_ema_rescaled)
    last_morning_ema = subid_mema.dttm_obs.max()
    last_morning_ema_rescaled = subid_mema.loc[subid_mema.dttm_obs==last_morning_ema].norm_date.values[0]
    #print(last_morning_ema)
    #print(last_morning_ema_rescaled)
    # Lapse
    subid_lapse = lapse_summary_df.query("subid==@id").copy()

    memas_in_study_period = len(mema_days[mema_days<=min(89,int(last_morning_ema_rescaled))])
    if int(last_morning_ema_rescaled)>0:
        responsiveness = memas_in_study_period/(min(89,int(last_morning_ema_rescaled))-int(first_morning_ema_rescaled)+1)
    else:
        responsiveness = 0
    # Make a subid info df
    subid_info_df = pd.DataFrame({'subid':id,
                                  'mema_count':len(mema_days),
                                 'last_morning_ema_day':int(last_morning_ema_rescaled),
                                 'first_morning_ema_day':int(first_morning_ema_rescaled),
                                 'responsiveness':responsiveness,
                                 },index=[0])
    subid_info_list.append(subid_info_df)
    subid_label_list = []
    # Loop over 90 day study period
    for day in range(90):
        if id==2 and day==47:
            print("hi")
        # Calculate the day of the week based on the day of the week at the beginning of the study
        day_of_week_num=(data_start_day_of_week+day)%7

        # Start time is either 0 (for day zero) or 4AM
        if day!=0:
            start=day+day_shift
        else:
            start=day
        # End of day
        end = day+day_shift+1
        mema_present = day in mema_days
        # Pre-allocate the EMA responses
        ema_2 = np.nan
        ema_3 = np.nan
        ema_4 = np.nan
        ema_5 = np.nan
        ema_6 = np.nan
        ema_7 = np.nan
        ema_8 = np.nan
        ema_9 = np.nan
        ema_10 = np.nan
        mema_time = np.nan
        # Handling for the event that there is a morning EMA available
        if mema_present:
            # Check for multiple
            subid_day_mema = subid_mema.query("rounded_date==@day").copy().sort_values(by="norm_date",ignore_index=True)
            count = subid_day_mema.shape[0]
            # TODO - consider handling here, what to do for more than one
            if count>1:
                print("multiple entries for subid {}, day {}".format(id,day))
                duplicate_ema_counter+=1
                for i in range(count):
                    print(subid_day_mema.iloc[i].time_of_day)
            mema_day = subid_day_mema.iloc[0]   
            # Special cases for missing data
            # if id==135 and day==0:
            #     mema_day = subid_day_mema.iloc[2]
            # if id==215 and day==0:
            #     mema_day = subid_day_mema.iloc[1]
            mema_time_of_day = mema_day.time_of_day
            mema_arrival = mema_day.norm_date
            mema_time = mema_arrival
            # Re-assign values of EMA responses
            ema_1 = mema_day.ema_1
            ema_2 = mema_day.ema_2_scaled
            ema_3 = mema_day.ema_3_scaled
            ema_4 = mema_day.ema_4_scaled
            ema_5 = mema_day.ema_5_scaled
            ema_6 = mema_day.ema_6_scaled
            ema_7 = mema_day.ema_7_scaled
            ema_8 = mema_day.ema_8_scaled
            ema_9 = mema_day.ema_9_scaled
            ema_10 = mema_day.ema_10_scaled

        # Default values for these lapse booleans is false
        # True values are assigned by evaluating each possible lapse case
        lapse_before_mema = False
        lapse_after_mema = False
        lapse_bool = False
        lapse_start_time = np.nan
        lapse_end_time = np.nan

        # Note, could refactor so we don't search this lapse list for every iteration
        for idx,row in subid_lapse.iterrows():
            lapse_start = row.norm_date
            lapse_start_rounded = row.start_rounded_date
            lapse_duration_raw = row.day_lapse_duration
            lapse_duration = 0 if np.isnan(lapse_duration_raw) else lapse_duration_raw
            lapse_end = lapse_start+lapse_duration
            # Check for case when start time is nan
            # 5/17 - no longer relevant. Special cases have been given a start time

            # Lapses broken out into 10 cases based on start times relative to day start/mema arrival/day end
            # Can be refactored for greater efficiency
            if lapse_end <= start or lapse_start >= end:
                # Cases independent of mema presence
                # If the lapse starts after the day window, no need to do anything
                # If the lapse ends before the day window, no need to do anything
                continue
            if mema_present:
                # 8 cases
                # Case 1 - Starts before day, ends before mema
                if lapse_start < start and lapse_end < mema_arrival:
                    lapse_before_mema = True
                    lapse_start_time = lapse_start
                    lapse_end_time = lapse_end
                # Case 2 - Starts before day, ends after mema
                elif lapse_start < start and lapse_end >= mema_arrival:
                    lapse_before_mema = True
                    lapse_after_mema = True
                    lapse_start_time = lapse_start
                    lapse_end_time = lapse_end
                # Case 3 - Starts before day, ends after day (previous case logic encompasses this one)
                # Case 4 - Starts before mema, ends before mema
                elif lapse_start >=start and lapse_start < mema_arrival and lapse_end < mema_arrival:
                    lapse_before_mema = True
                    lapse_start_time = lapse_start
                    lapse_end_time = lapse_end
                # Case 5 - Starts before mema, ends after mema
                elif lapse_start >=start and lapse_start < mema_arrival and lapse_end>=mema_arrival:
                    lapse_before_mema = True
                    lapse_after_mema = True
                    lapse_start_time = lapse_start
                    lapse_end_time = lapse_end
                # Case 6 - Starts before mema, ends after day (previous case logic encompasses this one)
                # Case 7 - Starts after mema, ends after mema
                elif lapse_start >= mema_arrival:
                    lapse_after_mema = True
                    lapse_start_time = lapse_start
                    lapse_end_time = lapse_end
                else:
                    print("error categorizing on day {} for subid {}".format(day,id))
                # Case 8 - Starts after mema, ends after day (previous case logic encompasses this one)
                lapse_bool = lapse_before_mema or lapse_after_mema
            else:
                # 4 cases
                # need to choose convention on lapse_before_mema and lapse_after_mema for these cases
                # Case 1 - Starts before day, ends during day
                if lapse_start < start and lapse_end < end:
                    lapse_bool = True
                    lapse_start_time = lapse_start
                    lapse_end_time = lapse_end
                # Case 2 - Starts before day, ends after day
                elif lapse_start < start and lapse_end >= end:
                    lapse_bool = True
                    lapse_start_time = lapse_start
                    lapse_end_time = lapse_end
                # Cases 3 and 4 - Starts during day and ends during day or goes into next day
                elif lapse_start >= start and lapse_start <= end:
                    lapse_bool = True
                    lapse_start_time = lapse_start
                    lapse_end_time = lapse_end
                else:
                    print("error categorizing on day {} for subid {}".format(day,id))

        # Construct a dataframe for this entry
        subid_day_label = pd.DataFrame({'subid':id,'day':day, #'ema_1':ema_1,
                                        'ema_2':ema_2,'ema_3':ema_3,'ema_4':ema_4,'ema_5':ema_5,
                                        'ema_6':ema_6,'ema_7':ema_7,'ema_8':ema_8,'ema_9':ema_9,'ema_10':ema_10,
                                        'lapse_before_mema':lapse_before_mema,'lapse_after_mema':lapse_after_mema,'lapse':lapse_bool,
                                        'mema_arrival':mema_time, 'lapse_start':lapse_start_time,'lapse_end':lapse_end_time,
                                        'day_of_week_num':day_of_week_num},index=[0])
        # Convert from boolean to numeric
        subid_day_label['lapse_before_mema']=subid_day_label['lapse_before_mema'].astype(int)
        subid_day_label['lapse_after_mema']=subid_day_label['lapse_after_mema'].astype(int)
        subid_day_label['lapse']=subid_day_label['lapse'].astype(int)
        #subid_day_label = pd.get_dummies(subid_day_label,columns=['day_of_week_num'],drop_first=)
        subid_label_list.append(subid_day_label)
    subid_df = pd.concat(subid_label_list,ignore_index=True)
    subid_df = pd.get_dummies(subid_df,columns = ['day_of_week_num'],drop_first=False)
    label_list.append(subid_df)
label_df = pd.concat(label_list,ignore_index=True)
label_df.to_csv('/Users/eric/repos/aud/data/day_labels.csv',index=False)
info_df = pd.concat(subid_info_list,ignore_index=True)
print(duplicate_ema_counter)
info_df.to_csv('/Users/eric/repos/aud/data/subject_info.csv',index=False)

In [ ]:
#429 cases of duplicate responses when blank EMAs are not removed, down to 368 after blank EMAs are removed during preprocessing

In [ ]:
print(info_df.responsiveness.max())
display(info_df[info_df.responsiveness<0.3])

In [ ]:
sns.histplot(data=info_df,x='responsiveness',bins=20)

In [ ]:
label_df.head()

In [ ]:
day_data = pd.read_csv('/Users/eric/repos/aud/data/day_labels.csv')
day_data_p = pd.read_csv('/Users/eric/repos/aud/data/day_labels_pre_exclusion.csv')
day_data.head()

In [ ]:
display(day_data.query("lapse_before_mema==1 & lapse_after_mema==0"))

In [ ]:
print(day_data.lapse.sum()/day_data.shape[0])
print(day_data.lapse.sum())
print(day_data_p.lapse.sum()/day_data_p.shape[0])
print(day_data_p.lapse.sum())